# HW4 - Notebook 1: Data and features

This notebook shows the data and the features behind my strategy, and checks them for looking ahead.
The reusable code lives in two files next to this notebook:

- [`data.py`](data.py): loads `equities_top2000.duckdb` with the course helpers (`load_data`, `load_membership`).
- [`features.py`](features.py): builds monthly features and the 90-day target (`python features.py`).

Both reuse functions from `Lectures/modeling/course_data.py`; each file says exactly which ones.

In [1]:
# Libraries
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display
import sys

# Course setup
course_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file())
modeling_dir = course_root / 'Lectures/modeling'
if str(modeling_dir) not in sys.path:
    sys.path.insert(0, str(modeling_dir))

# My modules
# read the database directly for hand checks
import duckdb
# database path + experiment settings
from data import DATABASE, experiment
from features import PANEL_PATH, NUMERIC_FEATURES, build_panel, check_timing

# show the experiment settings
experiment

{'database': 'equities_top2000.duckdb',
 'development_start': '2010-01-01',
 'development_end': '2015-12-31',
 'buffer_end': '2016-03-31',
 'test_start': '2016-04-01',
 'test_end': '2026-09-30',
 'holding_calendar_days': 90,
 'initial_cash': 1000000,
 'risk_free_hurdle': 0.02}

## 1. Build or load the feature panel

One row per **eligible stock per month-end signal date**: the stock is in that quarter's top 2000 list and has
at least 63 complete daily returns. Features use data through the signal close; the trade fills at the **next session's close**.

In [2]:
# Reuse the saved panel if it exists. Otherwise build it and save it.
if PANEL_PATH.exists():
    panel = pd.read_parquet(PANEL_PATH)
else:
    panel, _ = build_panel()
    PANEL_PATH.parent.mkdir(exist_ok=True)
    panel.to_parquet(PANEL_PATH, index=False)

# The market calendar needed by the timing checks below
# Converted to nanosecond datetimes to match the panel's date columns.
with duckdb.connect(str(DATABASE), read_only=True) as con:
    calendar = pd.DatetimeIndex(pd.to_datetime(
        con.execute('SELECT date FROM calendar ORDER BY date').fetchdf().date).astype('datetime64[ns]'))

# Size of the panel and display a few columns:
#   date: signal date
#   entry_date: next session
#   label_end : exit date
#   target    : actual 90-day return from entry to exit
print(f'{len(panel):,} rows x {panel.shape[1]} columns; {panel.symbol.nunique():,} stocks; {panel.date.nunique()} signal dates')
display(panel[['date', 'symbol', 'sector', 'momentum_63', 'volatility_63', 'net_margin',
               'entry_date', 'label_end', 'target']].head())

576,800 rows x 52 columns; 3,810 stocks; 317 signal dates


,date,symbol,sector,momentum_63,volatility_63,net_margin,entry_date,label_end,target
0,2000-04-28,A,Healthcare,0.301837,1.384914,0.058326,2000-05-01,2000-07-31,-0.572686
1,2000-04-28,AAON,Industrials,0.278756,0.504028,NaN,2000-05-01,2000-07-31,0.095238
2,2000-04-28,AAPL,Technology,0.223309,0.722974,0.078105,2000-05-01,2000-07-31,-0.182545
3,2000-04-28,ABCB,Financial Services,-0.003817,0.219014,NaN,2000-05-01,2000-07-31,-0.068966
4,2000-04-28,ABEO,Healthcare,-0.134487,1.682655,NaN,2000-05-01,2000-07-31,-0.324324


## 2. Coverage by period

Periods and every period's purpose

In [3]:
# label each row with its period. pd.cut puts each date into a bin:
period = pd.cut(panel.date, pd.to_datetime(['1999-12-31', '2009-12-31', '2015-12-31', '2016-03-31', '2026-09-30']),
                labels=['lookback 2000-09', 'development 2010-15', 'buffer 2016Q1', 'test 2016Q2-2026Q3'])

# Per period: number of rows, number of month-ends, and the share of rows with a finished 90-day return.
coverage = panel.groupby(period, observed=True).agg(
    rows=('symbol', 'size'), months=('date', 'nunique'), labeled_share=('target', lambda t: t.notna().mean()))
coverage['stocks_per_month'] = (coverage.rows / coverage.months).round().astype(int)
coverage['labeled_share'] = coverage.labeled_share.map('{:.1%}'.format)    # show as a percentage
display(coverage)

,rows,months,labeled_share,stocks_per_month
date,,,,
lookback 2000-09,181455,117,100.0%,1551
development 2010-15,141562,72,100.0%,1966
buffer 2016Q1,5980,3,99.8%,1993
test 2016Q2-2026Q3,247803,125,97.7%,1982


Unlabeled test rows are signals whose 90-day exit falls after Sep 30, 2026, or stocks with no quote within
5 sessions of maturity. They stay in the panel and can't be used for training.

## 3. Missing features

Missing values are **not** filled here. The model's imputer learns its fill values from training rows only (Notebook 2).

In [4]:
# development rows only (2010-2015)
dev = panel.loc[panel.date.between('2010-01-01', '2015-12-31')]
# .isna().mean() = share of rows where each feature is missing (True counts as 1, False as 0).
display(dev[NUMERIC_FEATURES].isna().mean().map('{:.1%}'.format).rename('missing share').to_frame())

,missing share
momentum_21,0.0%
momentum_63,0.0%
momentum_252_21,3.0%
volatility_63,0.0%
drawdown_252,3.3%
momentum_126,1.0%
log_dollar_volume_63,1.8%
log_universe_rank,0.0%
net_margin,2.1%
operating_roa,0.2%


## 4. Look-ahead checks on every row

`check_timing` (in `features.py`) stops with an error if any row breaks a timing rule.

In [5]:
# Five rules: next-session entry, financials already filed, universe list already
# published, exit at least 90 days after entry, and no exit after Sep 30, 2026. Raises an error if any fails.
check_timing(panel, calendar)

Timing checks passed: next-session entry, filed financials, known universe, >=90-day exits.


## 5. Hand check: recompute one label and one feature from raw SQL

The panel's numbers come from the course helpers. Here I recompute one row independently, straight from the
`prices` table, to confirm the entry/exit dates, the 90-day return, and a trailing feature.

In [6]:
SYMBOL, SIGNAL = 'AAPL', '2012-11-30'
# The panel's row for this stock and signal date
row = panel.loc[panel.symbol.eq(SYMBOL) & panel.date.eq(SIGNAL)].iloc[0]

# Read AAPL's full adjusted-price history straight from the database
with duckdb.connect(str(DATABASE), read_only=True) as con:
    px = con.execute('SELECT date, adj_close FROM prices WHERE symbol = ? ORDER BY date', [SYMBOL]).fetchdf()
px['date'] = pd.to_datetime(px.date)
px = px.set_index('date').adj_close

# Enter at the next session's close and exit at the first close on/ after entry + 90 calendar days.
# first trading day after the signal
entry_date = px.index[px.index > SIGNAL][0]
# after 90 calendar days 
due = entry_date + pd.Timedelta(days=90)
# first trading day on/ after that
exit_date = px.index[px.index >= due][0]
# 90-day return
hand_target = px[exit_date] / px[entry_date] - 1

# 63 session momentum 
history = px.loc[:SIGNAL]
# today vs 63 sessions earlier
hand_momentum_63 = history.iloc[-1] / history.iloc[-64] - 1

# Side by side: my hand calculation vs the panel.
check = pd.DataFrame({
    'by hand': [entry_date.date(), round(px[entry_date], 4), due.date(), exit_date.date(),
                round(px[exit_date], 4), round(hand_target, 6), round(hand_momentum_63, 6)],
    'panel':   [row.entry_date.date(), round(row.entry_price, 4), None, row.label_end.date(),
                round(row.exit_price, 4), round(row.target, 6), round(row.momentum_63, 6)],
}, index=['entry date', 'entry price', '90 days later', 'exit date', 'exit price', '90-day return', 'momentum_63'])
display(check)

# Stop with an error if anything differs
assert row.entry_date == entry_date and row.label_end == exit_date
assert np.isclose(hand_target, row.target) and np.isclose(hand_momentum_63, row.momentum_63)
print('Hand check matches the panel.')

,by hand,panel
entry date,2012-12-03,2012-12-03
entry price,17.71,17.71
90 days later,2013-03-03,None
exit date,2013-03-04,2013-03-04
exit price,12.75,12.75
90-day return,-0.280068,-0.280068
momentum_63,-0.127407,-0.127407


Hand check matches the panel.


Signal Friday Nov 30, 2012 → entry Monday Dec 3. Ninety calendar days later is Sunday Mar 3, 2013, so the exit is
the next session, Monday Mar 4. AAPL really did fall about 28% over that winter.